In [2]:
import requests, re
from urllib.parse import quote
requests.packages.urllib3.disable_warnings()

In [3]:
BASE_URL = 'https://mft.iqor.com'
USERNAME = 'Iberdrola'
PASSWORD = 'KaC#9eta'

In [4]:
import requests, re
from datetime import datetime
requests.packages.urllib3.disable_warnings()

BASE_URL = 'https://mft.iqor.com'
USERNAME = 'Iberdrola'
PASSWORD = 'KaC#9eta'

session = requests.Session()
session.verify = False
session.auth = (USERNAME, PASSWORD)   # <-- send auth on EVERY request
session.get(BASE_URL + '/files', timeout=15)

def list_dir(path):
    r = session.get(BASE_URL + path, timeout=15)
    print(f'\n=== {path} ===')
    files, folders = [], []
    for line in r.text.strip().splitlines()[1:]:
        parts = line.split()
        if len(parts) < 9:
            continue
        perms, _, owner, group, size, month, day, time, *name_parts = parts
        name = ' '.join(name_parts)
        if name in ('.', '..'):
            continue
        is_dir = perms.startswith('d')
        entry = {'name': name, 'size': int(size), 'date': f'{month} {day} {time}', 'type': 'DIR' if is_dir else 'FILE'}
        (folders if is_dir else files).append(entry)
    for f in folders:
        print(f"  [DIR ]  {f['name']}")
    for f in files:
        print(f"  [FILE]  {f['name']:55s}  {f['size']:>12,} bytes  {f['date']}")
    return files, folders

list_dir('/Report/Survey%20Reports/')
list_dir('/Report/Survey%20Reports/NSE/')
list_dir('/Report/Survey%20Reports/CMP/')



=== /Report/Survey%20Reports/ ===

=== /Report/Survey%20Reports/NSE/ ===
  [FILE]  NSE Daily Survey Report_20260413.csv                           57,305 bytes  Apr 14 07:00:30
  [FILE]  NSE Daily Survey Report_20260414.csv                           37,438 bytes  Apr 15 07:00:40
  [FILE]  NSE Daily Survey Report_20260415.csv                           37,936 bytes  Apr 16 07:00:49
  [FILE]  NSE Daily Survey Report_20260416.csv                           36,102 bytes  Apr 17 07:00:13
  [FILE]  NSE Daily Survey Report_20260417.csv                           36,069 bytes  Apr 18 07:00:30
  [FILE]  NSE Daily Survey Report_20260420.csv                           45,268 bytes  Apr 21 07:00:15
  [FILE]  NSE Daily Survey Report_20260421.csv                           38,818 bytes  Apr 22 07:00:26
  [FILE]  NSE Daily Survey Report_20260422.csv                           43,313 bytes  Apr 23 07:00:14
  [FILE]  NSE Daily Survey Report_20260423.csv                           42,399 bytes  Apr 24 07:00:46

([{'name': 'CMP Daily Survey Report_20260413.csv',
   'size': 31661,
   'date': 'Apr 14 07:00:28',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260414.csv',
   'size': 33110,
   'date': 'Apr 15 07:00:38',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260415.csv',
   'size': 27674,
   'date': 'Apr 16 07:00:24',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260416.csv',
   'size': 16866,
   'date': 'Apr 17 07:00:59',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260417.csv',
   'size': 17123,
   'date': 'Apr 18 07:01:15',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260420.csv',
   'size': 19254,
   'date': 'Apr 21 07:00:15',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260421.csv',
   'size': 21367,
   'date': 'Apr 22 07:00:26',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260422.csv',
   'size': 20809,
   'date': 'Apr 23 07:00:33',
   'type': 'FILE'},
  {'name': 'CMP Daily Survey Report_20260423.csv

In [5]:
import pandas as pd
from io import StringIO
from urllib.parse import quote

def get_latest_file(folder_path):
    # List folder and filter Daily Survey Reports (not Triage)
    files, _ = list_dir(folder_path)
    reports = [f for f in files if 'Daily Survey Report_' in f['name'] and 'Triage' not in f['name']]
    
    # Sort by date embedded in filename (YYYYMMDD)
    reports.sort(key=lambda f: f['name'].split('_')[-1].replace('.csv', ''))
    latest = reports[-1]
    print(f"Latest: {latest['name']}  ({latest['date']})")
    
    # Download
    encoded_name = quote(latest['name'])
    r = session.get(BASE_URL + folder_path + encoded_name, timeout=30)
    
    # Parse company and date from filename
    # e.g. "NSE Daily Survey Report_20260415.csv"
    parts = latest['name'].replace('.csv', '').split('_')
    company = parts[0].split()[0]          # NSE / CMP / RGE
    date    = pd.to_datetime(parts[-1], format='%Y%m%d')
    
    df = pd.read_csv(StringIO(r.text))
    df['company'] = company
    df['report_date'] = date
    print(f"  Shape: {df.shape}  |  Columns: {df.columns.tolist()[:6]}")
    return df

df_cmp = get_latest_file('/Report/Survey%20Reports/CMP/')
df_nse = get_latest_file('/Report/Survey%20Reports/NSE/')
df_rge = get_latest_file('/Report/Survey%20Reports/RGE/')



=== /Report/Survey%20Reports/CMP/ ===
  [FILE]  CMP Daily Survey Report_20260413.csv                           31,661 bytes  Apr 14 07:00:28
  [FILE]  CMP Daily Survey Report_20260414.csv                           33,110 bytes  Apr 15 07:00:38
  [FILE]  CMP Daily Survey Report_20260415.csv                           27,674 bytes  Apr 16 07:00:24
  [FILE]  CMP Daily Survey Report_20260416.csv                           16,866 bytes  Apr 17 07:00:59
  [FILE]  CMP Daily Survey Report_20260417.csv                           17,123 bytes  Apr 18 07:01:15
  [FILE]  CMP Daily Survey Report_20260420.csv                           19,254 bytes  Apr 21 07:00:15
  [FILE]  CMP Daily Survey Report_20260421.csv                           21,367 bytes  Apr 22 07:00:26
  [FILE]  CMP Daily Survey Report_20260422.csv                           20,809 bytes  Apr 23 07:00:33
  [FILE]  CMP Daily Survey Report_20260423.csv                           15,737 bytes  Apr 24 07:00:23
  [FILE]  CMP Daily Survey Report_

In [6]:
display(df_cmp.head(5))

,ID,Name,Date_Time,Work_Group,InteractionID,Phone_Number,Survey_Name,CSAT1,NPS,I_C,C_K,FCR,Call_Reason,Survey_Status_Count,Survey_Status,company,report_date
0,34795431,allie.frank,04/27/2026 07:37:56,CMP.USUT.CS.RESCRCL,700908311232,2075310673,CMP IQR Survey w/ NPS,5.0,5.0,NaN,#,#,6.0,NaN,ABANDONED,CMP,2026-04-27
1,39610865,shaylin.wallace,04/27/2026 07:38:24,CMP.USUT.CS.RESCRCL,700908312391,2076894846,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED,CMP,2026-04-27
2,56540384,timothy.cleary,04/27/2026 07:38:59,CMP.USUT.CS.RESCRCL,700908311796,2076923406,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED,CMP,2026-04-27
3,34795447,tiffany.white5,04/27/2026 07:40:10,CMP.USUT.CS.RESCRCL,700908311985,6039301979,CMP IQR Survey w/ NPS,4.0,4.0,#,#,#,1.0,6.0,COMPLETED,CMP,2026-04-27
4,39116229,stephanie.guillen,04/27/2026 07:40:38,CMP.USUT.CS.RESCRCL,700908311984,2073807234,CMP IQR Survey w/ NPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABANDONED,CMP,2026-04-27
